# parserInvestfunds — выгрузка ПИФов с pro.investfunds.ru **без фильтров**

Что поменялось против прежней редакции:

| было | стало |
|---|---|
| фильтры: тип (ОПИФ/БПИФ/ИПИФ) × категория × специализация, 6 книг | фильтров нет, отмечено всё → **один** набор файлов |
| периоды: полугодия, список дат руками | **кварталы** назад от `сегодня - DAYS_BACK` (X в хардкоде) |
| сшивка через Excel/xlwings + `strip_inner_headers` | скачанные файлы читаются по очереди в **один `DataFrame df`** |
| результат: `akcii_investfundsОПИФЫ.xlsx` и ещё 5 книг | результат: `df` в памяти (сохранение — последняя ячейка, по желанию) |

Excel на машине больше не нужен — вся сборка в pandas. Файлы падают в папку
рядом с кодом (`./investfunds_xlsx`), а не туда, где лежит venv.

**Плата за отказ от фильтров:** в выгрузке сайта нет колонок «тип фонда» и
«категория» — раньше их заменяло имя книги (`akcii_…ОПИФЫ`). В общем `df`
останутся только колонки сайта (Дата / Фонд / УК / Пай / СЧА); тип и категорию,
если понадобятся, придётся подтягивать отдельным справочником.

Порядок прогона: ячейка 1 (настройки) → 2 (утилиты) → 3 (скачивание, Selenium +
msedgedriver.exe) → 4 (сборка `df`) → 5 (сводка). Ячейку 4 можно крутить
отдельно, не перекачивая файлы.

In [2]:
# =============================================================================
# 1. Настройки и периоды
# =============================================================================
import datetime
import os
import re
import time
from pathlib import Path

import pandas as pd

# ------------------------------------------------------------------ хардкод ---
DAYS_BACK = 2        # X: правый край самого свежего периода = сегодня - X дней
N_QUARTERS = 22      # сколько кварталов тянуть назад (22 ~ с 3 кв. 2021)

LOGIN = "baxa"
PASSWORD = "mozart"

DOWNLOAD_TIMEOUT = 600   # сек на одну выгрузку: без фильтров файл крупный
SETTLE_AFTER_DATES = 2   # сек на пересборку таблицы под новый диапазон
RETRIES = 1              # повторов на период, если выгрузка не доехала
SKIP_EXISTING = True     # уже скачанный период повторно не тянуть
KEEP_SOURCE_COL = False  # оставить в df служебную колонку с именем файла

# --------------------------------------------------------------------- пути ---
# Папка с кодом, а не с venv. В Jupyter у ячейки нет __file__, поэтому берём cwd
# ядра: для ноутбука это его собственная папка. В .py-скрипте замените строку на
# BASE_DIR = Path(__file__).resolve().parent
BASE_DIR = Path.cwd()
DOWNLOAD_DIR = BASE_DIR / "investfunds_xlsx"
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

# msedgedriver.exe: сначала рядом с кодом, потом по известным местам.
# Selenium тут работает только с этим драйвером, поэтому путь ищем, а не угадываем.
DRIVER_CANDIDATES = (
    BASE_DIR / "msedgedriver.exe",
    Path(r"C:\Users\tmp_SKosyrkov\Downloads\py_ex\general_0\msedgedriver.exe"),
    Path.home() / "Downloads" / "msedgedriver.exe",
)


def resolve_driver(candidates=DRIVER_CANDIDATES) -> Path:
    """Первый существующий путь к msedgedriver.exe или понятная ошибка."""
    for p in candidates:
        if Path(p).is_file():
            return Path(p)
    raise FileNotFoundError(
        "msedgedriver.exe не найден. Положите его рядом с кодом "
        f"({BASE_DIR}) или допишите путь в DRIVER_CANDIDATES. Искал:\n  "
        + "\n  ".join(str(p) for p in candidates)
    )


# ------------------------------------------------------- кварталы от «сегодня»
SITE_FMT = "%d.%m.%Y"   # формат календаря на сайте


def _q_start(d: datetime.date) -> datetime.date:
    """Первое число квартала, в который попала дата."""
    return d.replace(month=3 * ((d.month - 1) // 3) + 1, day=1)


def quarter_periods(anchor: datetime.date, n: int):
    """
    n кварталов назад от anchor: [(начало, конец), ...], от свежего к старому.

    Самый свежий период обрезан по anchor — квартал ещё не закончился. Остальные
    полные: шаг «первое число квартала минус день» сам попадает на конец
    предыдущего квартала, так что длину квартала считать не нужно.
    """
    out, cur = [], anchor
    for _ in range(n):
        start = _q_start(cur)
        out.append((start, cur))
        cur = start - datetime.timedelta(days=1)
    return out


ANCHOR_DATE = datetime.date.today() - datetime.timedelta(days=DAYS_BACK)
PERIODS = quarter_periods(ANCHOR_DATE, N_QUARTERS)

print(f"сегодня {datetime.date.today():{SITE_FMT}}, якорь (-{DAYS_BACK} дн.) "
      f"{ANCHOR_DATE:{SITE_FMT}}, периодов {len(PERIODS)}")
print(f"файлы кладём в {DOWNLOAD_DIR}")
for s, e in PERIODS:
    print(f"  {s:{SITE_FMT}} - {e:{SITE_FMT}}")

сегодня 21.10.2026, якорь (-2 дн.) 19.10.2026, периодов 22
файлы кладём в c:\Users\tmp_SKosyrkov\Downloads\py_ex\peers_vs_funds_2\investfunds_xlsx
  01.10.2026 - 19.10.2026
  01.07.2026 - 30.09.2026
  01.04.2026 - 30.06.2026
  01.01.2026 - 31.03.2026
  01.10.2025 - 31.12.2025
  01.07.2025 - 30.09.2025
  01.04.2025 - 30.06.2025
  01.01.2025 - 31.03.2025
  01.10.2024 - 31.12.2024
  01.07.2024 - 30.09.2024
  01.04.2024 - 30.06.2024
  01.01.2024 - 31.03.2024
  01.10.2023 - 31.12.2023
  01.07.2023 - 30.09.2023
  01.04.2023 - 30.06.2023
  01.01.2023 - 31.03.2023
  01.10.2022 - 31.12.2022
  01.07.2022 - 30.09.2022
  01.04.2022 - 30.06.2022
  01.01.2022 - 31.03.2022
  01.10.2021 - 31.12.2021
  01.07.2021 - 30.09.2021


In [6]:
# =============================================================================
# 2. Утилиты файловой системы: ждём докачку, раскладываем файлы по периодам
# =============================================================================
PARTIAL_SUFFIXES = (".crdownload", ".tmp", ".partial")


def dir_snapshot(folder) -> set:
    return set(os.listdir(folder))


def wait_for_new_download(folder, before: set, timeout: int = DOWNLOAD_TIMEOUT) -> Path:
    """Ждёт появления нового полностью догруженного файла и возвращает его путь."""
    deadline = time.time() + timeout
    last_size = -1
    while time.time() < deadline:
        fresh = [f for f in os.listdir(folder)
                 if f not in before and not f.endswith(PARTIAL_SUFFIXES)]
        if fresh:
            path = max((Path(folder) / f for f in fresh),
                       key=lambda p: p.stat().st_mtime)
            size = path.stat().st_size
            if size > 0 and size == last_size:   # два замера подряд совпали
                return path
            last_size = size
        time.sleep(0.7)
    raise TimeoutError(f"Новый файл не появился в {folder} за {timeout} с")


def period_path(start, fin, suffix=".xlsx") -> Path:
    """
    Имя под период: investfunds_20260701_20260930.xlsx.

    Детерминированное имя вместо «Выгрузка (7).xls» даёт две вещи: сортировка
    по имени = сортировка по времени, и повторный прогон того же периода
    перезатирает файл, а не плодит копии.
    """
    return DOWNLOAD_DIR / f"investfunds_{start:%Y%m%d}_{fin:%Y%m%d}{suffix}"


def existing_period_file(start, fin):
    """Уже скачанный файл периода (любое расширение) или None."""
    hits = sorted(DOWNLOAD_DIR.glob(f"investfunds_{start:%Y%m%d}_{fin:%Y%m%d}.*"))
    return hits[0] if hits else None

In [10]:
# =============================================================================
# 3. Скачивание: Selenium + msedgedriver.exe, фильтры НЕ применяются
# =============================================================================
from selenium import webdriver
from selenium.webdriver.edge.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    NoSuchElementException,
    StaleElementReferenceException,
    TimeoutException,
)

# type_check_list  - тип фонда (ОПИФ/БПИФ/ИПИФ/ЗПИФ)
# cat_check_list   - категория (акции, облигации, ...)
# spec_check_list  - специализация внутри категории
FILTER_CONTAINERS = ("type_check_list", "cat_check_list", "spec_check_list")


def filter_state(driver, container_id):
    """(отмечено, всего) по контейнеру — чтобы глазами видеть, что фильтра нет."""
    boxes = driver.find_elements(
        By.CSS_SELECTOR, f"#{container_id} input[type=checkbox]")
    on = 0
    for b in boxes:
        try:
            on += bool(b.is_selected())
        except StaleElementReferenceException:
            pass
    return on, len(boxes)


def ensure_all_checked(driver, containers=FILTER_CONTAINERS, settle=0.6, attempts=2):
    """
    «Не фильтровать» = отмечено всё. На свежей странице так и есть, но профиль
    Edge (--user-data-dir) умеет помнить прошлый выбор, поэтому недостающие галки
    доставляем сами — иначе молча получим урезанную выгрузку.

    Порядок контейнеров важен: тип -> категории -> специализации, нижние списки
    перерисовываются обработчиком верхних. Клик делаем через JS .click(), а не
    присваиванием b.checked = true: присваивание не шлёт событий, и inline
    onclick=setspecs()/setuk() не выполнится.
    """
    report = {}
    for cid in containers:
        for _ in range(attempts):
            on, total = filter_state(driver, cid)
            if total and on == total:
                break
            driver.execute_script(
                "document.querySelectorAll(arguments[0] + ' input[type=checkbox]')"
                ".forEach(function (b) { if (!b.checked) b.click(); });",
                "#" + cid,
            )
            time.sleep(settle)
        report[cid] = filter_state(driver, cid)   # в норме (N, N) по каждому
    return report


DRIVER_PATH = resolve_driver()
print("драйвер:", DRIVER_PATH)

edge_options = webdriver.EdgeOptions()
# edge_options.add_argument("--headless=new")
edge_options.add_argument("--window-size=1920,1080")
edge_options.add_argument("--disable-blink-features=AutomationControlled")
edge_options.add_argument("--use-system-proxy-resolver")
user_data_dir = os.path.expanduser("~") + r"\AppData\Local\Microsoft\User Data"
edge_options.add_argument(f"--user-data-dir={user_data_dir}")
edge_options.add_argument("--profile-directory=Default")
edge_options.add_experimental_option(
    "prefs",
    {
        # качаем рядом с кодом, а не в папку venv и не в системные «Загрузки»
        "download.default_directory": str(DOWNLOAD_DIR),
        "download.prompt_for_download": False,
        "download.directory_upgrade": True,
        "safebrowsing.enabled": True,
    },
)

driver = webdriver.Edge(
    options=edge_options,
    service=Service(executable_path=str(DRIVER_PATH)),
)
wait = WebDriverWait(driver, 20)

saved, failed = [], []
try:
    driver.get("https://pro.investfunds.ru/")
    driver.get("https://pro.investfunds.ru/quotes/")

    try:
        ln = driver.find_element(By.ID, "loginName")
        ln.click()
        ln.send_keys(LOGIN)
        pw = driver.find_element(By.ID, "loginPass")
        pw.click()
        pw.send_keys(PASSWORD)
        driver.find_element(
            By.CSS_SELECTOR, "input.header-user-login-submit[type='submit']"
        ).click()
        time.sleep(2)
    except NoSuchElementException:
        # профиль Edge помнит сессию — формы логина на странице просто нет
        print("форма логина не найдена, похоже уже залогинены")

    wait.until(EC.presence_of_element_located((By.ID, "csv")))
    print("фильтры (отмечено/всего):", ensure_all_checked(driver))

    for i, (start, fin) in enumerate(PERIODS, 1):
        tag = f"[{i}/{len(PERIODS)}] {start:{SITE_FMT}}-{fin:{SITE_FMT}}"

        have = existing_period_file(start, fin)
        if SKIP_EXISTING and have:
            print(f"{tag}: уже лежит, пропуск ({have.name})")
            saved.append(have)
            continue

        for attempt in range(RETRIES + 1):
            try:
                st_dat = driver.find_element(By.ID, "calendar-1")
                fin_dat = driver.find_element(By.ID, "calendar-2")

                st_dat.send_keys(Keys.CONTROL + "a")
                st_dat.send_keys(Keys.BACKSPACE)
                st_dat.send_keys(f"{start:{SITE_FMT}}")

                fin_dat.send_keys(Keys.CONTROL + "a")
                fin_dat.send_keys(Keys.BACKSPACE)
                fin_dat.send_keys(f"{fin:{SITE_FMT}}")
                fin_dat.send_keys(Keys.ESCAPE)   # закрыть всплывающий календарь

                time.sleep(SETTLE_AFTER_DATES)   # дать таблице пересобраться

                before = dir_snapshot(DOWNLOAD_DIR)
                driver.find_element(By.ID, "csv").click()
                got = wait_for_new_download(DOWNLOAD_DIR, before)

                dst = period_path(start, fin, got.suffix.lower() or ".xlsx")
                os.replace(got, dst)
                saved.append(dst)
                print(f"{tag}: {dst.name}, {dst.stat().st_size / 1024:.0f} КБ")
                break
            except (TimeoutError, TimeoutException) as err:
                # один период не доехал — остальные всё равно забираем
                print(f"{tag}: попытка {attempt + 1} мимо — {err}")
                if attempt == RETRIES:
                    failed.append((f"{start:{SITE_FMT}}", f"{fin:{SITE_FMT}}", repr(err)))
finally:
    driver.quit()

print(f"\nскачано {len(saved)}, провалов {len(failed)}")
for f in failed:
    print("  провал:", f)

драйвер: C:\Users\tmp_SKosyrkov\Downloads\py_ex\general_0\msedgedriver.exe
фильтры (отмечено/всего): {'type_check_list': (4, 4), 'cat_check_list': (18, 18), 'spec_check_list': (88, 88)}
[1/22] 01.10.2026-19.10.2026: investfunds_20261001_20261019.csv, 377 КБ
[2/22] 01.07.2026-30.09.2026: investfunds_20260701_20260930.csv, 1947 КБ
[3/22] 01.04.2026-30.06.2026: investfunds_20260401_20260630.csv, 1738 КБ
[4/22] 01.01.2026-31.03.2026: investfunds_20260101_20260331.csv, 1512 КБ
[5/22] 01.10.2025-31.12.2025: investfunds_20251001_20251231.csv, 1715 КБ
[6/22] 01.07.2025-30.09.2025: investfunds_20250701_20250930.csv, 1693 КБ
[7/22] 01.04.2025-30.06.2025: investfunds_20250401_20250630.csv, 1463 КБ
[8/22] 01.01.2025-31.03.2025: investfunds_20250101_20250331.csv, 1391 КБ
[9/22] 01.10.2024-31.12.2024: investfunds_20241001_20241231.csv, 1513 КБ
[10/22] 01.07.2024-30.09.2024: investfunds_20240701_20240930.csv, 1498 КБ
[11/22] 01.04.2024-30.06.2024: investfunds_20240401_20240630.csv, 1340 КБ
[12/22] 01

In [11]:
def clean_uploaded_df(path_to_that_csv, start_date, finish_date):
    df = pd.read_excel(path_to_that_csv).dropna(axis=0)

    def fix_all_encoding(text):
        if isinstance(text, str):
            try:
                # Сначала пробуем стандартный перевод из Latin-1 в CP1251
                return text.encode("latin1").decode("cp1251")
            except UnicodeEncodeError:
                # Если упало на спецсимволах (тире, кавычки), подменяем их вручную через cp1252
                return text.encode("cp1252", errors="replace").decode(
                    "cp1251", errors="replace"
                )
            except Exception:
                return text
        return text

    df = df.map(
        fix_all_encoding
    )  # Если pandas старый, используй df.applymap(fix_encoding)
    df.columns = pd.Series(df.loc[1])
    df = df[2:]
    num = int(
        df["Пай"].count()
    )  # в файле есть мусор в строке дат, смотрим на другие - например, "пай"
    df = df[:num]
    df = df[df["УК"] != "ТКБ Инвестмент Партнерс"]
    df = df[df["СЧА"] != "Не раскрывается"]
    path_to_xlsx = rf"""C:/Users/tmp_SKosyrkov/Documents/files_for_peers_v_funds/test/funds_{start_date.replace('.', '')}_{finish_date.replace('.', '')}.xlsx"""
    df.to_excel(path_to_xlsx, index=False)
    return path_to_xlsx

In [13]:
# =============================================================================
# 4. Сборка одного DataFrame из скачанных файлов
#    (замена прежней связки xlwings + strip_inner_headers)
# =============================================================================
HEADER_VALUES = ("Дата", "Фонд", "УК", "Пай", "СЧА")
_WANT = {h.lower() for h in HEADER_VALUES}
_NAME_RE = re.compile(r"investfunds_(\d{8})_(\d{8})")


def _norm(v) -> str:
    """Ячейка без nbsp, двойных пробелов и регистра — для сравнения со шапкой."""
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return ""
    return re.sub(r"\s+", " ", str(v).replace("\xa0", " ")).strip().lower()


def read_sheet(path) -> pd.DataFrame:
    """
    Первый лист как есть, без шапки. Движок выбирает pandas сам; кнопка XLS на
    сайте иногда отдаёт HTML-таблицу под расширением .xls — на ней read_excel
    падает, и выручает read_html.
    """
    path = Path(path)
    try:
        return pd.read_csv(path, header=None, dtype=object)
    except Exception as err_excel:
        tables = None
        for enc in ("utf-8", "cp1251"):
            try:
                tables = pd.read_html(path, encoding=enc)
                break
            except Exception:
                continue
        if not tables:
            raise RuntimeError(
                f"{path.name}: не читается ни как Excel ({err_excel}), ни как HTML"
            ) from err_excel
        t = max(tables, key=len)
        if not isinstance(t.columns, pd.RangeIndex):   # шапку забрал <th> — вернём строкой
            head = pd.DataFrame([list(t.columns)])
            t = pd.concat([head, t.set_axis(range(t.shape[1]), axis=1)], ignore_index=True)
        return t.set_axis(range(t.shape[1]), axis=1).astype(object)


def extract_block(df: pd.DataFrame, name: str):
    """
    То же, что прежние ANCHOR='C4' + Ctrl+A + strip_inner_headers, но в pandas:
    ищем строку-шапку, берём только её непустые столбцы (так отваливаются пустые
    A/B и заголовок выгрузки сверху), ниже выкидываем повторы шапки и пустые строки.

    Возвращает (кадр, убрано_шапок, убрано_пустых).
    """
    norm = df.apply(lambda col: col.map(_norm))

    hdr = None
    for i in range(len(norm)):
        # >=3 совпадений, а не все 5: страховка, если сайт переименует столбец
        if len(set(norm.iloc[i]) & _WANT) >= 3:
            hdr = i
            break
    if hdr is None:
        raise ValueError(f"{name}: не нашёл строку-шапку {HEADER_VALUES}")

    cols = [j for j in range(df.shape[1]) if norm.iat[hdr, j]]
    head = [re.sub(r"\s+", " ", str(df.iat[hdr, j]).replace("\xa0", " ")).strip()
            for j in cols]

    body = df.iloc[hdr + 1:, cols]
    if body.empty:
        return pd.DataFrame(columns=head), 0, 0

    bn = norm.iloc[hdr + 1:, cols]
    key = bn.apply(lambda r: "|".join(x for x in r if x), axis=1)  # ключ строки без пустых
    hdr_key = "|".join(x for x in norm.iloc[hdr, cols] if x)

    is_hdr = key == hdr_key
    is_empty = key == ""
    out = body.loc[~(is_hdr | is_empty)].copy()
    out.columns = head
    return out.reset_index(drop=True), int(is_hdr.sum()), int(is_empty.sum())


def to_num(s: pd.Series) -> pd.Series:
    """'1 234,56' / 1234.56 -> float; прочерки и пустые -> NaN."""
    t = (s.astype(str)
          .str.replace("\xa0", "", regex=False)
          .str.replace(" ", "", regex=False)
          .str.replace(",", ".", regex=False))
    return pd.to_numeric(t, errors="coerce")


def to_date(s: pd.Series) -> pd.Series:
    """
    В столбце мешаются строки 'дд.мм.гггг' и настоящие datetime (их распознал
    openpyxl). Формат для строк задаём явно: pd.to_datetime('01.02.2026') без
    формата — это 2 января, а не 1 февраля.
    """
    if pd.api.types.is_datetime64_any_dtype(s):
        return s
    out = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")
    as_str = s.map(lambda v: isinstance(v, str))
    if as_str.any():
        out.loc[as_str] = pd.to_datetime(
            s[as_str].str.strip(), format="%d.%m.%Y", errors="coerce")
    if (~as_str).any():
        out.loc[~as_str] = pd.to_datetime(s[~as_str], errors="coerce")
    return out


def typed(part: pd.DataFrame):
    """Даты -> datetime, Пай/СЧА -> float, текст -> без nbsp и двойных пробелов."""
    date_col = next(c for c in part.columns if _norm(c) == "дата")
    num_cols = [c for c in part.columns if _norm(c) in {"пай", "сча"}]

    part[date_col] = to_date(part[date_col])
    for c in num_cols:
        part[c] = to_num(part[c])
    for c in part.columns:
        if c in (date_col, *num_cols) or c.startswith("_"):
            continue
        part[c] = part[c].map(
            lambda v: re.sub(r"\s+", " ", v.replace("\xa0", " ")).strip()
            if isinstance(v, str) else v)
    return part, date_col, num_cols


def period_from_name(name: str):
    """(начало, конец) из имени investfunds_20260701_20260930.xlsx или None."""
    m = _NAME_RE.match(name)
    return tuple(pd.Timestamp(g) for g in m.groups()) if m else None


# ---------------------------------------------------------------- сборка ------
files = sorted(DOWNLOAD_DIR.glob("investfunds_*"))
if not files:
    raise FileNotFoundError(
        f"в {DOWNLOAD_DIR} нет файлов investfunds_* — сначала прогоните ячейку скачивания")

frames, suspicious = [], []
for f in files:
    part, n_hdr, n_empty = extract_block(read_sheet(f), f.name)
    part, date_col, _ = typed(part)
    part["_файл"] = f.name          # след источника; снимается ниже по KEEP_SOURCE_COL

    span = "пусто"
    if not part.empty and part[date_col].notna().any():
        lo, hi = part[date_col].min(), part[date_col].max()
        span = f"{lo:{SITE_FMT}}-{hi:{SITE_FMT}}"
        want = period_from_name(f.name)
        # Если календарь на сайте не применился, файл приедет с данными чужого
        # периода. Дальше дедуп такие копии молча склеит, поэтому ловим здесь.
        if want and (lo < want[0] or hi > want[1]):
            suspicious.append(f.name)
            span += " (!) вне периода из имени"

    frames.append(part)
    print(f"{f.name}: {len(part)} строк, {span} "
          f"(шапок убрано {n_hdr}, пустых {n_empty})")

layouts = {tuple(p.columns) for p in frames}
if len(layouts) > 1:
    raise ValueError("файлы с разным набором столбцов: "
                     + "; ".join(str(sorted(l)) for l in layouts))

df = pd.concat(frames, ignore_index=True)

DATE_COL = next(c for c in df.columns if _norm(c) == "дата")
NUM_COLS = [c for c in df.columns if _norm(c) in {"пай", "сча"}]
TXT_COLS = [c for c in df.columns if c not in (DATE_COL, *NUM_COLS, "_файл")]

no_date = int(df[DATE_COL].isna().sum())      # подвальные примечания выгрузки
df = df.loc[df[DATE_COL].notna()]

was = len(df)
df = df.drop_duplicates(subset=[c for c in df.columns if c != "_файл"])
dups = was - len(df)

df = df.sort_values([DATE_COL, *TXT_COLS], kind="stable").reset_index(drop=True)
if not KEEP_SOURCE_COL:
    df = df.drop(columns="_файл")

print(f"\nитог: {len(df)} строк, {len(df.columns)} столбцов; "
      f"без даты отброшено {no_date}, дублей снято {dups}")
if not df.empty:
    print(f"период {df[DATE_COL].min():{SITE_FMT}} - {df[DATE_COL].max():{SITE_FMT}}, "
          f"дат {df[DATE_COL].nunique()}, фондов {df[TXT_COLS[0]].nunique()}")
if suspicious:
    print("ВНИМАНИЕ: даты не применились в файлах — перекачайте их, "
          "иначе период потерян:\n  " + "\n  ".join(suspicious))
df.head()

RuntimeError: investfunds_20210701_20210930.csv: не читается ни как Excel ('utf-8' codec can't decode byte 0xc4 in position 0: invalid continuation byte), ни как HTML

In [ ]:
# =============================================================================
# 5. Что получилось (и, если нужно, сохранение)
# =============================================================================
print(df.dtypes.to_string(), "\n")
print("строк по годам:")
print(df.groupby(df[DATE_COL].dt.year).size().to_string(), "\n")
print("топ-5 УК по числу строк:")
print(df[TXT_COLS[-1]].value_counts().head().to_string())

# Рабочий результат — сам df. Сохранение по желанию:
# df.to_parquet(BASE_DIR / "investfunds_all.parquet")           # компактно и быстро
# df.to_excel(BASE_DIR / "investfunds_all.xlsx", index=False)   # > 1 048 575 строк не влезет